In [ ]:
import sys
sys.path.append("..")

import pandas as pd
from src.scoutai_common import get_engine, load_master_view, engineer_features, route_predict

engine = get_engine()
df = load_master_view(engine)
df = engineer_features(df)
df = route_predict(df)  # adds predicted_value, model_used

valid_gems = df[(df['predicted_value'] > df['current_market_value']) &
                (df['total_appearances'] >= 15) & (df['age'] > 15)].copy()
valid_gems['value_gap_eur'] = valid_gems['predicted_value'] - valid_gems['current_market_value']

def format_currency(x): return f"E{x:,.0f}"
cols_to_format = ['current_market_value', 'predicted_value', 'value_gap_eur']

pos_map = {
    'Goalkeeper': 'TOP 5 UNDERVALUED GOALKEEPERS',
    'Defender': 'TOP 5 UNDERVALUED DEFENDERS',
    'Attack': 'TOP 5 UNDERVALUED FORWARDS',
    'Midfield': 'TOP 5 UNDERVALUED MIDFIELDERS',
}

for pos, title in pos_map.items():
    print(f"\n--- {title} ---")
    data = valid_gems[valid_gems['position_group'] == pos].sort_values(by='value_gap_eur', ascending=False).head(5)
    if not data.empty:
        display_data = data[['player_name', 'age', 'model_used'] + cols_to_format].copy()
        for col in cols_to_format:
            display_data[col] = display_data[col].map(format_currency)
        print(display_data.to_string(index=False))
    else:
        print("No undervalued players found in this category.")